In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://rdm.example.com/'
admin_rdm_url = 'https://admin.rdm.example.com/'
idp_name_1 = 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
idp_name_2 = 'GakuNin RDM IdP'
idp_username_2 = None
idp_password_2 = None

weko_url = 'https://weko.example.com/'
weko_admin_email = None
weko_admin_password = None
weko_user_email = None
weko_user_password = None
weko_institution_name = 'GakuNin RDM IdP'
weko_index_name = 'Sample Index'
ignore_https_errors = False

# Migration前手順で作成された移行済みプロジェクト
rdm_project_name_w_metadata = 'TEST-Migration-20250906-METADATA'

project_name_prefix = 'TEST-WEKO-{}'
oauth_application_name_prefix = 'TEST-WEKO-APP-{}'
default_result_path = None
close_on_fail = False
transition_timeout = 60000
skip_failed_test = False
exclude_notebooks = []

# OAuthクライアント追加テスト用設定
is_staff = False
clear_apps = True

# WEKO SWORD API settings
weko_docker_compose_path = None
sword_mapping_id = 30002
sword_workflow_id = 1

weko_test_mode = 'direct'


In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for {idp_name_2}')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_2}')

if weko_admin_email is None:
    weko_admin_email = input('WEKO admin email: ')
if weko_admin_password is None:
    weko_admin_password = getpass('WEKO admin password: ')
if weko_user_email is None:
    weko_user_email = input('WEKO user email: ')
if weko_user_password is None:
    weko_user_password = getpass('WEKO user password: ')
if weko_institution_name is None:
    weko_institution_name = input('機関名 (例: GakuNin RDM IdP): ')
if weko_index_name is None:
    weko_index_name = input('WEKO index name: ')

timestamp = datetime.now().strftime('%Y%m%d-%H%M')
project_name_dashboard = project_name_prefix.format(timestamp + '-1')
oauth_application_name_dashboard = oauth_application_name_prefix.format(timestamp + '-PD')
project_name_files_tab = project_name_prefix.format(timestamp + '-2')
oauth_application_name_files_tab = oauth_application_name_prefix.format(timestamp + '-FT')


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


# Migration後 - WEKOアドオンによるメタデータ送信（取りまとめ）

- サブシステム名: アドオン
- ページ/アドオン: Metadata, WEKO
- 機能分類: Migration後の動作確認
- シナリオ名: 移行済みメタデータのWEKO送信

このNotebookは取りまとめNotebookであり、実際のテストは他のNotebookで行います。


In [ ]:
from datetime import datetime
import os
from scripts.papermillHelpers import gen_run_notebook


def make_result_dir(base_path):
    result_dir = os.path.join(base_path, 'notebooks')
    os.makedirs(result_dir, exist_ok=True)
    return result_dir


result_dir = make_result_dir(default_result_path)

run_notebook = gen_run_notebook(
    result_dir,
    transition_timeout,
    dict(
        rdm_url=rdm_url,
        admin_rdm_url=admin_rdm_url,
        idp_name_1=idp_name_1,
        idp_username_1=idp_username_1,
        idp_password_1=idp_password_1,
        # Migration: the metadata-bearing project was created pre-migration by idp_1
        # and is affiliated with idp_1's institution (Virginia Tech), where JAIRO Cloud
        # is enabled. The WEKO operations (OAuth app owner, addon-add, deposit) must run
        # as that same user, so the idp_2 slot is wired to idp_1.
        idp_name_2=idp_name_1,
        idp_username_2=idp_username_1,
        idp_password_2=idp_password_1,
        weko_url=weko_url,
        weko_admin_email=weko_admin_email,
        weko_admin_password=weko_admin_password,
        weko_user_email=weko_user_email,
        weko_user_password=weko_user_password,
        weko_institution_name=weko_institution_name,
        weko_index_name=weko_index_name,
        ignore_https_errors=ignore_https_errors,
    ),
    skip_failed_test,
    exclude_notebooks,
)

result_notebooks = []
result_dir


In [ ]:
import shutil

# WEKOアドオン関連Notebookはリポジトリ直下にあるため、実行ディレクトリへ複製して参照する
for _nb in ['テスト手順-WEKOアドオン-OAuthクライアント追加.ipynb', 'テスト手順-WEKOアドオン-アドオン追加.ipynb']:
    if not os.path.exists(_nb):
        shutil.copy(os.path.join('..', _nb), _nb)


## 「WEKOアドオン-OAuthクライアント追加」テストの実施

プロジェクトダッシュボードでのテスト用に、テスト「テスト手順-WEKOアドオン-OAuthクライアント追加」を実施する。

In [ ]:
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-OAuthクライアント追加.ipynb',
        dict(
            project_name=project_name_dashboard,
            oauth_application_name=oauth_application_name_dashboard,
            is_staff=is_staff,
            clear_apps=clear_apps,
        ),
    )
)
result_notebooks[-1]


## プロジェクトダッシュボードでのテスト用のSWORD Client登録

プロジェクトダッシュボードでのテスト用に、WEKOにSWORD Clientを登録する。OAuthクライアント作成時に生成されたclient_idを使用する。

In [ ]:
import json
import subprocess

addon_result_dir = os.path.join(result_dir, 'テスト手順-WEKOアドオン-OAuthクライアント追加')
oauth_info_path = os.path.join(addon_result_dir, 'weko_oauth_client.json')

with open(oauth_info_path) as f:
    oauth_info = json.load(f)
client_id_dashboard = oauth_info['client_id']
print(f'Loaded client_id: {client_id_dashboard}')

if weko_docker_compose_path:
    if weko_test_mode == 'workflow':
        reg_type = 'M.RegistrationType.WORKFLOW'
        reg_extra = f'workflow_id={sword_workflow_id},'
    else:
        reg_type = 'M.RegistrationType.DIRECT'
        reg_extra = ''
    code = f'''
from weko_swordserver.api import SwordClient
from weko_swordserver.models import SwordClientModel as M
obj = SwordClient.register(
    client_id="{client_id_dashboard}",
    registration_type_id={reg_type},
    mapping_id={sword_mapping_id},
    {reg_extra}
    active=True,
    duplicate_check=False,
)
print(f"Registered SWORD client ({{obj.registration_type}}): {{obj.client_id}}")
'''
    subprocess.run(
        ['docker', 'compose', '-f', weko_docker_compose_path, 'exec', '-T', 'web',
         'invenio', 'shell', '-c', code],
        check=True
    )
else:
    oauth_application_name = oauth_info['oauth_application_name']
    reg_type_label = 'WorkFlow' if weko_test_mode == 'workflow' else 'Direct'
    extra_instruction = '、WorkFlowはデフォルトアイテムタイプ(フル)' if weko_test_mode == 'workflow' else ''
    print(f'WEKO {weko_url} の Administration > SWORD API > JSON-LD の Create をクリックし、Application={oauth_application_name}, Registration Type={reg_type_label}{extra_instruction}、マッピングはデフォルトマッピング(フル)でSaveしてください。')
    assert input(prompt='実施したら OK とタイプしてください').lower() == 'ok'


## WEKOアドオンのテスト実施

アドオン追加およびメタデータ送信テストを実施する。以下の観点で確認を行う。

- 操作画面: プロジェクトダッシュボード / ファイルタブ
- 送信メタデータ: 根拠データのみ / 書誌データのみ / 根拠データ＋書誌データ

In [ ]:
# 移行済みプロジェクトへWEKOアドオンを接続し、移行済みメタデータをWEKOへ送信する
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-アドオン追加.ipynb',
        dict(
            project_name=rdm_project_name_w_metadata,
            oauth_application_name=oauth_application_name_dashboard,
            oauth_client_type=weko_test_mode,
        ),
        '-移行済みプロジェクト',
    )
)
result_notebooks.append(
    run_notebook(
        'Migration後テスト手順-WEKOアドオン.ipynb',
        dict(
            rdm_project_name_w_metadata=rdm_project_name_w_metadata,
            oauth_client_type=weko_test_mode,
        ),
    )
)
result_notebooks[-1]


終了処理を実施。


In [ ]:
!rm -fr {work_dir}
